# Regression model

In [1]:
import pandas as pd
import numpy as np
from scipy.linalg import qr
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
# Load all datasets
raw_allranks = pd.read_excel("Archive of OSF Storage\\Raw score data\\allranks_raw.xlsx")
raw_bronze = pd.read_excel("Archive of OSF Storage\\Raw score data\\bronze_raw.xlsx")
raw_gold = pd.read_excel("Archive of OSF Storage\\Raw score data\\gold_raw.xlsx")
raw_diamond = pd.read_excel("Archive of OSF Storage\\Raw score data\\diamond_raw.xlsx")
raw_gc = pd.read_excel("Archive of OSF Storage\\Raw score data\\gc_raw.xlsx")

diff_allranks = pd.read_excel("Archive of OSF Storage\\Difference score data\\allrank_diff.xlsx")
diff_bronze = pd.read_excel("Archive of OSF Storage\\Difference score data\\bronze_diff.xlsx")
diff_gold = pd.read_excel("Archive of OSF Storage\\Difference score data\\gold_diff.xlsx")
diff_diamond = pd.read_excel("Archive of OSF Storage\\Difference score data\\diamond_diff.xlsx")
diff_gc = pd.read_excel("Archive of OSF Storage\\Difference score data\\GC_diff.xlsx")

In [3]:
# copilot was used for the remove_multicollinear_columns_qr function, as I was unsure how to replicate the R logic in Python.
def remove_multicollinear_columns_qr(X, tol=1e-10):
    """
    Remove multicollinear columns using QR decomposition with column pivoting.

    Parameters
    ----------
    X : pandas.DataFrame
        Feature matrix. All columns should be numeric.
    tol : float
        Threshold for detecting linear dependence from the diagonal of R.

    Returns
    -------
    X_reduced : pandas.DataFrame
        DataFrame containing only linearly independent columns.
    kept_columns : list[str]
        Column names kept after QR selection.
    dropped_columns : list[str]
        Column names removed as collinear.
    """
    if not isinstance(X, pd.DataFrame):
        raise TypeError('X must be a pandas DataFrame')

    X_numeric = X.apply(pd.to_numeric, errors='coerce')
    valid = X_numeric.dropna(axis=1, how='any')
    if valid.shape[1] == 0:
        raise ValueError('No fully numeric columns available after coercion')

    matrix = valid.to_numpy(dtype=float)
    _, r, piv = qr(matrix, mode='economic', pivoting=True)
    diag = np.abs(np.diag(r))

    if diag.size == 0:
        kept_idx = np.array([], dtype=int)
    else:
        threshold = tol * diag[0]
        kept_idx = np.where(diag > threshold)[0]

    kept_columns = valid.columns[piv[kept_idx]].tolist()
    dropped_columns = [c for c in valid.columns if c not in kept_columns]

    return valid.loc[:, kept_columns].copy(), kept_columns, dropped_columns


In [4]:
def find_best_ntree_regression(X, y, max_trees=1000, step=10, random_state=123):
    
    results = []
    max_features = max(1, X.shape[1] // 3)

    for n in range(100, max_trees + 1, step):
        rf = RandomForestRegressor(
            n_estimators=n,
            max_features=max_features,
            bootstrap=True,
            oob_score=True,
            n_jobs=-1,
            random_state=random_state,
        )
        rf.fit(X, y)

        oob_pred = rf.oob_prediction_
        mse = mean_squared_error(y, oob_pred)
        r2 = r2_score(y, oob_pred)

        results.append({
            'ntree': n,
            'oob_mse': mse,
            'oob_r2': r2,
        })

    return pd.DataFrame(results).sort_values('oob_mse').reset_index(drop=True)

In [5]:
# The authors seems to have tested with a step of 1, however even with a step of 10 the code the optimaization took an hour to run. As such we have opted to make a compromise and use a step of 5.
def find_best_ntree_regression(X, y, max_trees=1000, step=5, random_state=123):
    results = []

    max_features = max(1, X.shape[1] // 3) # This replicates the R default of mtry = p/3 for regression, where p is the number of predictors.

    for n in range(100, max_trees + 1, step):
        rf = RandomForestRegressor(
            n_estimators=n,
            max_features=max_features,
            bootstrap=True,
            oob_score=True,
            n_jobs=-1,
            random_state=random_state
        )

        rf.fit(X, y)

        oob_pred = rf.oob_prediction_
        mse = mean_squared_error(y, oob_pred)
        r2 = r2_score(y, oob_pred)

        results.append({
            "ntree": n,
            "oob_mse": mse,
            "oob_r2": r2
        })

    return pd.DataFrame(results).sort_values("oob_mse")

In [6]:
# Process all datasets with QR screening and ntree optimization

target_col = 'diff.goals.ag'
datasets = [
    ('raw_allranks', raw_allranks),
    ('raw_bronze', raw_bronze),
    ('raw_gold', raw_gold),
    ('raw_diamond', raw_diamond),
    ('raw_gc', raw_gc),
    ('diff_allranks', diff_allranks),
    ('diff_bronze', diff_bronze),
    ('diff_gold', diff_gold),
    ('diff_diamond', diff_diamond),
    ('diff_gc', diff_gc),
]

all_results = {}
best_results_summary = []

for dataset_name, df in datasets:

    candidate_features = df.drop(columns=[target_col])

    # Apply QR-based multicollinearity removal
    X_reduced, kept_columns, dropped_columns = remove_multicollinear_columns_qr(candidate_features, tol=1e-10)

    if dropped_columns:
        print(f"Dropped columns: {dropped_columns}")

    # Run ntree optimization
    ntree_results = find_best_ntree_regression(X_reduced, df[target_col], max_trees=1000, step=100, random_state=123)

    best_row = ntree_results.iloc[0]
    best_ntree = int(best_row['ntree'])
    best_mse = best_row['oob_mse']
    best_r2 = best_row['oob_r2']

    # Store detailed results
    all_results[dataset_name] = {
        'X_reduced': X_reduced,
        'y': df[target_col],
        'kept_columns': kept_columns,
        'dropped_columns': dropped_columns,
        'ntree_results': ntree_results,
        'best_ntree': best_ntree,
        'best_mse': best_mse,
        'best_r2': best_r2,
    }

    # Store compact summary row
    best_results_summary.append({
        'Dataset': dataset_name,
        'Type': dataset_name.split('_', 1)[0],
        'N_Features_Original': candidate_features.shape[1],
        'N_Features_QR': len(kept_columns),
        'N_Dropped': len(dropped_columns),
        'Best_ntree': best_ntree,
        'OOB_MSE': best_mse,
        'OOB_R2': best_r2,
    })

# Create summary DataFrame
summary_df = pd.DataFrame(best_results_summary)
print("\n" + "=" * 80)
print("Summary of ntree optimization across all datasets")
print("=" * 80)
print(summary_df.to_string(index=False))

# Save summary to CSV
summary_df.to_csv('ntree_optimization_summary.csv', index=False)


Summary of ntree optimization across all datasets
      Dataset Type  N_Features_Original  N_Features_QR  N_Dropped  Best_ntree  OOB_MSE   OOB_R2
 raw_allranks  raw                   28             28          0         900 3.612278 0.746857
   raw_bronze  raw                   28             28          0        1000 3.909363 0.792502
     raw_gold  raw                   28             28          0         900 3.519303 0.743003
  raw_diamond  raw                   28             28          0        1000 3.659033 0.725310
       raw_gc  raw                   27             27          0        1000 4.052708 0.713095
diff_allranks diff                   26             26          0        1000 2.295546 0.839098
  diff_bronze diff                   26             26          0        1000 2.991057 0.841057
    diff_gold diff                   26             26          0        1000 2.433282 0.822310
 diff_diamond diff                   26             26          0        1000 2.36586

In [7]:
# Simplified rfPermute-style helpers with improved efficiency and methodology
# Note: This is an approximation of the R rfPermute workflow used in the Rocket League paper.
# Exact reproduction would require matching R randomForest/rfPermute's OOB permutation importance
# implementation, which scikit-learn implements differently.



# Reduced computational parameters to balance feasibility with rigor
n_permutations = 10        # 10 permutations of the response variable for null distribution
                   # The paper does not explicitly report the number of rfPermute permutations
                   # used for regression models; 10 is a feasible approximation
n_repeats = 2      # 2 repeats of feature permutation for observed importance stability
n_repeats_NULL = 1 # 1 repeat for null models (null distribution already comes from response permutations)

# Updated optimal ntree values from paper's Figure 6
optimal_ntrees = {
    'raw_allranks': 954,
    'raw_bronze': 856,
    'raw_gold': 380,
    'raw_diamond': 1000,
    'raw_gc': 1000,
    'diff_allranks': 986,
    'diff_bronze': 961,
    'diff_gold': 895,
    'diff_diamond': 989,
    'diff_gc': 991,
}

def fit_dataset_model(dataset_name, random_state=123):
    if dataset_name not in all_results:
        raise ValueError(f"Unknown dataset_name: {dataset_name}")

    dataset_result = all_results[dataset_name]
    X = dataset_result['X_reduced']
    y = dataset_result['y']
    kept_columns = dataset_result['kept_columns']
    dropped_columns = dataset_result['dropped_columns']
    ntree = optimal_ntrees[dataset_name]

    rf = RandomForestRegressor(
        n_estimators=ntree,
        max_features=max(1, X.shape[1] // 3),
        bootstrap=True,
        oob_score=True,
        n_jobs=-1,
        random_state=random_state,
    )
    rf.fit(X, y)
    oob_mse = mean_squared_error(y, rf.oob_prediction_)
    oob_r2 = r2_score(y, rf.oob_prediction_)

    return {
        'dataset': dataset_name,
        'ntree': ntree,
        'X': X,
        'y': y,
        'rf': rf,
        'kept_columns': kept_columns,
        'dropped_columns': dropped_columns,
        'model_summary': {
            'Dataset': dataset_name,
            'OOB_MSE': oob_mse,
            'OOB_R2': oob_r2,
        },
    }


In [8]:
# Fit raw_allranks model
raw_allranks_fit = fit_dataset_model('raw_allranks')
pd.DataFrame([raw_allranks_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,raw_allranks,3.612215,0.746861


In [9]:
# Fit raw_bronze model
raw_bronze_fit = fit_dataset_model('raw_bronze')
pd.DataFrame([raw_bronze_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,raw_bronze,3.909717,0.792483


In [10]:
# Fit raw_gold model
raw_gold_fit = fit_dataset_model('raw_gold')
pd.DataFrame([raw_gold_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,raw_gold,3.530396,0.742193


In [11]:
# Fit raw_diamond model
raw_diamond_fit = fit_dataset_model('raw_diamond')
pd.DataFrame([raw_diamond_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,raw_diamond,3.659033,0.72531


In [12]:
# Fit raw_gc model
raw_gc_fit = fit_dataset_model('raw_gc')
pd.DataFrame([raw_gc_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,raw_gc,4.052708,0.713095


In [13]:
# Fit diff_allranks model
diff_allranks_fit = fit_dataset_model('diff_allranks')
pd.DataFrame([diff_allranks_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,diff_allranks,2.295987,0.839067


In [14]:
# Fit diff_bronze model
diff_bronze_fit = fit_dataset_model('diff_bronze')
pd.DataFrame([diff_bronze_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,diff_bronze,2.993299,0.840938


In [15]:
# Fit diff_gold model
diff_gold_fit = fit_dataset_model('diff_gold')
pd.DataFrame([diff_gold_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,diff_gold,2.434942,0.822189


In [16]:
# Fit diff_diamond model
diff_diamond_fit = fit_dataset_model('diff_diamond')
pd.DataFrame([diff_diamond_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,diff_diamond,2.366678,0.82233


In [17]:
# Fit diff_gc model
diff_gc_fit = fit_dataset_model('diff_gc')
pd.DataFrame([diff_gc_fit['model_summary']])

,Dataset,OOB_MSE,OOB_R2
0,diff_gc,2.613332,0.814993


In [22]:
from sklearn.inspection import permutation_importance
# The paper uses %IncMSE, which is the mean increase in MSE from permuting a feature, expressed as a percentage of the baseline MSE.
def calculate_pct_inc_mse(rf, X, y, n_repeats=n_repeats, random_state=123):
    perm_result = permutation_importance(
        rf,
        X,
        y,
        n_repeats=n_repeats,
        scoring='neg_mean_squared_error',
        random_state=random_state,
        n_jobs=1, # Some issues were showing up with parallel jobs so we set the n_jobs parameter to 1
    )

    baseline_mse = mean_squared_error(y, rf.predict(X))
    mean_imp_pct = (perm_result.importances_mean / baseline_mse) * 100
    std_imp_pct = (perm_result.importances_std / baseline_mse) * 100

    return mean_imp_pct, std_imp_pct

def calculate_p_values(X, y, ntree, observed_importance, n_perm=n_permutations,
                       n_repeats_null=n_repeats_NULL, random_state=123):
    
    rng = np.random.default_rng(random_state)
    null_importances = np.zeros((n_perm, X.shape[1]))

    for i in range(n_perm):
        y_perm = pd.Series(rng.permutation(y.to_numpy()), index=y.index, name=y.name)

        rf_null = RandomForestRegressor(
            n_estimators=ntree,
            max_features=max(1, X.shape[1] // 3),
            bootstrap=True,
            oob_score=True,
            n_jobs=1,
            random_state=int(rng.integers(0, 123456789)),
        )
        rf_null.fit(X, y_perm)

        null_scores, _ = calculate_pct_inc_mse(
            rf_null,
            X,
            y_perm,
            n_repeats=n_repeats_null,
            random_state=int(rng.integers(0, 123456789)),
        )
        null_importances[i] = null_scores

    p_values = ((null_importances >= observed_importance).sum(axis=0) + 1) / (n_perm + 1)
    return p_values, null_importances

def run_dataset_permutation(fit_result, n_permutations=n_permutations, n_repeats=n_repeats,
                           n_repeats_null=n_repeats_NULL, random_state=123):
    
    X = fit_result['X']
    y = fit_result['y']
    rf = fit_result['rf']
    dataset_name = fit_result['dataset']

    importance_mean, importance_sd = calculate_pct_inc_mse(
        rf,
        X,
        y,
        n_repeats=n_repeats,
        random_state=random_state,
    )

    p_values, null_importances = calculate_p_values(
        X,
        y,
        fit_result['ntree'],
        observed_importance=importance_mean,
        n_perm=n_permutations,
        n_repeats_null=n_repeats_null,
        random_state=random_state
    )

    importance_df = pd.DataFrame({
        'Dataset': fit_result['dataset'],
        'Feature': X.columns,
        'PctIncMSE': importance_mean,
        'PctIncMSE_SD': importance_sd,
        'p_value': p_values,
        'Significant': p_values < 0.05,
    }).sort_values('PctIncMSE', ascending=False).reset_index(drop=True)

    return {
        'dataset': fit_result['dataset'],
        'importance_df': importance_df,
        'null_importances': null_importances,
    }


In [24]:
# Permutation importance for raw_allranks
raw_allranks_perm = run_dataset_permutation(raw_allranks_fit)
raw_allranks_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,raw_allranks,shots.conceded.ag,1850.263875,12.271431,0.090909,False
1,raw_allranks,percentage.behind.ball,1102.897025,3.947985,0.090909,False
2,raw_allranks,shots.ag,852.424646,6.003760,0.090909,False
3,raw_allranks,saves.ag,379.480175,1.982039,0.090909,False
4,raw_allranks,percentage.defensive.third,92.714580,0.140014,0.272727,False
5,raw_allranks,percentage.offensive.third,89.666191,0.640198,0.272727,False
6,raw_allranks,percentage.supersonic.speed,46.304545,0.008106,1.000000,False
7,raw_allranks,avg.boost.amount,44.843290,0.406231,1.000000,False
8,raw_allranks,count.stolen.small.pads.ag,43.227024,0.147183,1.000000,False
9,raw_allranks,percentage.high.in.air,40.179037,0.197711,1.000000,False


In [25]:
# Permutation importance for raw_bronze
raw_bronze_perm = run_dataset_permutation(raw_bronze_fit)
raw_bronze_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,raw_bronze,shots.conceded.ag,1306.292179,16.414213,0.090909,False
1,raw_bronze,shots.ag,1024.831578,3.892172,0.090909,False
2,raw_bronze,percentage.behind.ball,955.178044,37.836368,0.090909,False
3,raw_bronze,saves.ag,138.294458,1.315266,0.090909,False
4,raw_bronze,percentage.offensive.third,106.714605,2.597718,0.090909,False
5,raw_bronze,percentage.defensive.third,65.364044,0.258350,0.181818,False
6,raw_bronze,percentage.high.in.air,48.027082,0.597655,0.909091,False
7,raw_bronze,avg.boost.amount,39.270827,0.837032,1.000000,False
8,raw_bronze,100.boost.time.ag,35.495894,1.255514,1.000000,False
9,raw_bronze,amount.stolen.ag,33.158650,0.195727,1.000000,False


In [26]:
# Permutation importance for raw_gold
raw_gold_perm = run_dataset_permutation(raw_gold_fit)
raw_gold_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,raw_gold,shots.conceded.ag,1474.507641,8.879353,0.090909,False
1,raw_gold,shots.ag,867.712222,12.553205,0.090909,False
2,raw_gold,percentage.behind.ball,842.419271,0.734196,0.090909,False
3,raw_gold,saves.ag,302.686394,1.927253,0.090909,False
4,raw_gold,percentage.offensive.third,75.313720,1.313365,0.090909,False
5,raw_gold,percentage.defensive.third,66.309143,0.452544,0.454545,False
6,raw_gold,avg.boost.amount,37.814160,0.686602,1.000000,False
7,raw_gold,percentage.high.in.air,35.846062,0.342112,1.000000,False
8,raw_gold,count.stolen.small.pads.ag,35.029034,0.439201,1.000000,False
9,raw_gold,0.boost.time.ag,32.379589,0.093047,1.000000,False


In [27]:
# Permutation importance for raw_diamond
raw_diamond_perm = run_dataset_permutation(raw_diamond_fit)
raw_diamond_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,raw_diamond,percentage.behind.ball,1165.073785,16.309555,0.090909,False
1,raw_diamond,shots.conceded.ag,1084.648689,7.326956,0.090909,False
2,raw_diamond,shots.ag,823.481738,7.113972,0.090909,False
3,raw_diamond,saves.ag,265.651473,1.417945,0.090909,False
4,raw_diamond,percentage.defensive.third,77.525816,0.285258,0.181818,False
5,raw_diamond,percentage.offensive.third,64.712109,0.322860,0.454545,False
6,raw_diamond,percentage.high.in.air,44.834993,0.009649,1.000000,False
7,raw_diamond,demos.inflicted.ag,39.916640,0.372224,0.454545,False
8,raw_diamond,0.boost.time.ag,37.555856,0.166045,1.000000,False
9,raw_diamond,avg.boost.amount,34.845170,0.452197,1.000000,False


In [28]:
# Permutation importance for raw_gc
raw_gc_perm = run_dataset_permutation(raw_gc_fit)
raw_gc_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,raw_gc,percentage.behind.ball,1349.949306,3.315082,0.090909,False
1,raw_gc,shots.conceded.ag,944.803606,7.045629,0.090909,False
2,raw_gc,shots.ag,640.135198,1.345423,0.090909,False
3,raw_gc,saves.ag,207.202710,1.734110,0.090909,False
4,raw_gc,percentage.defensive.third,70.902732,0.267288,0.181818,False
5,raw_gc,percentage.offensive.third,61.667991,0.390053,0.090909,False
6,raw_gc,percentage.on.ground,49.753416,0.329245,1.000000,False
7,raw_gc,demos.inflicted.ag,46.509029,1.281621,0.272727,False
8,raw_gc,amount.stolen.ag,34.314415,0.526886,1.000000,False
9,raw_gc,count.stolen.small.pads.ag,33.550516,0.991822,1.000000,False


In [29]:
# Permutation importance for diff_allranks
diff_allranks_perm = run_dataset_permutation(diff_allranks_fit)
diff_allranks_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,diff_allranks,diff.shots.ag,3954.558088,7.821875,0.090909,False
1,diff_allranks,diff.percentage.behind.ball.ag,1840.358843,13.436739,0.090909,False
2,diff_allranks,diff.saves.ag,856.973273,5.949917,0.090909,False
3,diff_allranks,diff.percentage.offensive.third.ag,127.081145,0.061478,0.090909,False
4,diff_allranks,diff.percentage.defensive.third.ag,122.783426,1.129009,0.090909,False
5,diff_allranks,diff.percentage.high.in.air.ag,85.175917,0.533599,0.272727,False
6,diff_allranks,diff.avg.boost.amount.ag,59.720226,0.528978,1.000000,False
7,diff_allranks,diff.count.stolen.small.pads.ag,56.236633,0.026826,1.000000,False
8,diff_allranks,diff.0.boost.time.ag,53.364228,0.074348,1.000000,False
9,diff_allranks,diff.demos.inflicted.ag,52.556078,0.457581,1.000000,False


In [30]:
# Permutation importance for diff_bronze
diff_bronze_perm = run_dataset_permutation(diff_bronze_fit)
diff_bronze_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,diff_bronze,diff.shots.ag,3071.479853,19.522416,0.090909,False
1,diff_bronze,diff.percentage.behind.ball.ag,1475.042003,34.382664,0.090909,False
2,diff_bronze,diff.saves.ag,300.871289,3.614323,0.090909,False
3,diff_bronze,diff.percentage.offensive.third.ag,113.286184,3.303079,0.090909,False
4,diff_bronze,diff.percentage.defensive.third.ag,76.548440,1.357552,0.090909,False
5,diff_bronze,diff.percentage.high.in.air.ag,73.961720,0.129673,0.181818,False
6,diff_bronze,diff.avg.boost.amount.ag,64.829849,1.838657,0.272727,False
7,diff_bronze,diff.count.stolen.small.pads.ag,42.301675,0.056852,1.000000,False
8,diff_bronze,diff.amount.stolen.ag,39.706902,0.623002,1.000000,False
9,diff_bronze,diff.count.powerslide.ag,37.761058,0.617528,1.000000,False


In [31]:
# Permutation importance for diff_gold
diff_gold_perm = run_dataset_permutation(diff_gold_fit)
diff_gold_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,diff_gold,diff.shots.ag,3652.699484,14.068622,0.090909,False
1,diff_gold,diff.percentage.behind.ball.ag,1359.656845,7.289888,0.090909,False
2,diff_gold,diff.saves.ag,635.287939,5.950659,0.090909,False
3,diff_gold,diff.percentage.offensive.third.ag,109.820342,2.548802,0.090909,False
4,diff_gold,diff.percentage.defensive.third.ag,99.089293,0.523311,0.090909,False
5,diff_gold,diff.percentage.high.in.air.ag,79.150126,0.448362,0.272727,False
6,diff_gold,diff.count.stolen.small.pads.ag,50.416320,0.173313,1.000000,False
7,diff_gold,diff.amount.stolen.ag,47.217675,0.277867,1.000000,False
8,diff_gold,diff.avg.boost.amount.ag,45.211860,0.662053,1.000000,False
9,diff_gold,diff.0.boost.time.ag,43.753831,0.623461,1.000000,False


In [32]:
# Permutation importance for diff_diamond
diff_diamond_perm = run_dataset_permutation(diff_diamond_fit)
diff_diamond_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,diff_diamond,diff.shots.ag,3075.816331,24.762066,0.090909,False
1,diff_diamond,diff.percentage.behind.ball.ag,1970.031956,1.024382,0.090909,False
2,diff_diamond,diff.saves.ag,631.885739,2.552353,0.090909,False
3,diff_diamond,diff.percentage.offensive.third.ag,112.246234,0.314825,0.090909,False
4,diff_diamond,diff.percentage.defensive.third.ag,97.343219,0.787180,0.090909,False
5,diff_diamond,diff.percentage.high.in.air.ag,61.924022,0.086640,0.727273,False
6,diff_diamond,diff.0.boost.time.ag,51.468599,0.229215,1.000000,False
7,diff_diamond,diff.demos.inflicted.ag,50.310993,1.014453,0.454545,False
8,diff_diamond,diff.avg.boost.amount.ag,47.941044,0.158946,1.000000,False
9,diff_diamond,diff.count.stolen.small.pads.ag,45.540481,0.171237,1.000000,False


In [33]:
# Permutation importance for diff_gc
diff_gc_perm = run_dataset_permutation(diff_gc_fit)
diff_gc_perm['importance_df'].head(10)

,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,diff_gc,diff.shots.ag,2399.298822,12.085053,0.090909,False
1,diff_gc,diff.percentage.behind.ball.ag,2255.538982,26.928791,0.090909,False
2,diff_gc,diff.saves.ag,496.561840,3.738632,0.090909,False
3,diff_gc,diff.percentage.offensive.third.ag,116.894025,1.861278,0.090909,False
4,diff_gc,diff.percentage.defensive.third.ag,88.645330,1.050944,0.090909,False
5,diff_gc,diff.0.boost.time.ag,60.033199,0.150574,0.727273,False
6,diff_gc,diff.demos.inflicted.ag,50.239771,0.174769,0.545455,False
7,diff_gc,diff.avg.boost.amount.ag,44.521546,0.155201,1.000000,False
8,diff_gc,diff.amount.stolen.ag,41.130837,0.885340,1.000000,False
9,diff_gc,diff.count.stolen.small.pads.ag,39.081071,0.764129,1.000000,False


In [34]:
import json
from pathlib import Path

fit_results = [
    raw_allranks_fit,
    raw_bronze_fit,
    raw_gold_fit,
    raw_diamond_fit,
    raw_gc_fit,
    diff_allranks_fit,
    diff_bronze_fit,
    diff_gold_fit,
    diff_diamond_fit,
    diff_gc_fit,
]

permutation_results = [
    raw_allranks_perm,
    raw_bronze_perm,
    raw_gold_perm,
    raw_diamond_perm,
    raw_gc_perm,
    diff_allranks_perm,
    diff_bronze_perm,
    diff_gold_perm,
    diff_diamond_perm,
    diff_gc_perm,
]

model_summary_df = pd.DataFrame([result['model_summary'] for result in fit_results])
all_importance_df = pd.concat([result['importance_df'] for result in permutation_results], ignore_index=True)
significance_summary = all_importance_df.groupby('Dataset')['Significant'].sum().reset_index(name='N_Significant_p05')
model_summary_df = model_summary_df.merge(significance_summary, on='Dataset', how='left')

results_dir = Path('rfpermute_saved_results')
results_dir.mkdir(exist_ok=True)

model_summary_path = results_dir / 'rfpermute_model_summary.csv'
importance_path = results_dir / 'rfpermute_importance_results.csv'
json_path = results_dir / 'rfpermute_results.json'

model_summary_df.to_csv(model_summary_path, index=False)
all_importance_df.to_csv(importance_path, index=False)

results_bundle = {
    'model_summary': model_summary_df.to_dict(orient='records'),
    'feature_importance': all_importance_df.to_dict(orient='records'),
    'datasets': {},
}

with json_path.open('w', encoding='utf-8') as file_handle:
    json.dump(results_bundle, file_handle, indent=2)

print(model_summary_df.to_string(index=False))

all_importance_df.head(20)

      Dataset  OOB_MSE   OOB_R2  N_Significant_p05
 raw_allranks 3.612215 0.746861                  0
   raw_bronze 3.909717 0.792483                  0
     raw_gold 3.530396 0.742193                  0
  raw_diamond 3.659033 0.725310                  0
       raw_gc 4.052708 0.713095                  0
diff_allranks 2.295987 0.839067                  0
  diff_bronze 2.993299 0.840938                  0
    diff_gold 2.434942 0.822189                  0
 diff_diamond 2.366678 0.822330                  0
      diff_gc 2.613332 0.814993                  0


,Dataset,Feature,PctIncMSE,PctIncMSE_SD,p_value,Significant
0,raw_allranks,shots.conceded.ag,1850.263875,12.271431,0.090909,False
1,raw_allranks,percentage.behind.ball,1102.897025,3.947985,0.090909,False
2,raw_allranks,shots.ag,852.424646,6.003760,0.090909,False
3,raw_allranks,saves.ag,379.480175,1.982039,0.090909,False
4,raw_allranks,percentage.defensive.third,92.714580,0.140014,0.272727,False
5,raw_allranks,percentage.offensive.third,89.666191,0.640198,0.272727,False
6,raw_allranks,percentage.supersonic.speed,46.304545,0.008106,1.000000,False
7,raw_allranks,avg.boost.amount,44.843290,0.406231,1.000000,False
8,raw_allranks,count.stolen.small.pads.ag,43.227024,0.147183,1.000000,False
9,raw_allranks,percentage.high.in.air,40.179037,0.197711,1.000000,False


In [ ]:
#commented out since this is for loading cached results
#import json
#from pathlib import Path

#results_dir = Path('rfpermute_saved_results')
#model_summary_cached = pd.read_csv(results_dir / 'rfpermute_model_summary.csv')
#importance_cached = pd.read_csv(results_dir / 'rfpermute_importance_results.csv')

#with (results_dir / 'rfpermute_results.json').open('r', encoding='utf-8') as file_handle:
#    results_cached = json.load(file_handle)